In [1]:
"""
==================================================================================
 CUSTOMER RETENTION INTELLIGENCE PLATFORM - SECONDARY DATASET (Olist)
 Phase 2: Data Cleaning -> olist_order_payments_dataset.csv (for Tableau dashboard)
==================================================================================
 Purpose : Clean the payment transaction table for use in Tableau, joined to
           olist_orders_dataset.csv via 'order_id'.

 IMPORTANT NOTES FOR A BI/DASHBOARD DATASET:
   1. Multiple rows can share the same order_id (split/multi-method payments)
      -> this is expected grain, not a duplicate to remove.
   2. A handful of rows have payment_type = 'not_defined' and
      payment_value = 0 -> these are genuinely unknown transactions in the
      source data, not something we can safely infer. They are relabeled to
      make them visible/filterable in Tableau rather than silently dropped.
==================================================================================
"""

import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 150)


def section(title: str) -> None:
    print("\n" + "=" * 90)
    print(f" {title}")
    print("=" * 90)


# ----------------------------------------------------------------------------------
# 1. LOAD RAW DATA
# ----------------------------------------------------------------------------------
section("1. LOAD RAW DATA")

RAW_PATH = "/Users/meetnakrani/Library/Mobile Documents/com~apple~CloudDocs/Customer-Retention-intelligence-Platform/DataSet/RAW/secondary Data/olist_order_payments_dataset.csv"
CLEANED_PATH = "/Users/meetnakrani/Library/Mobile Documents/com~apple~CloudDocs/Customer-Retention-intelligence-Platform/DataSet/cleaned/olist_order_payments_cleaned.csv"

df = pd.read_csv(RAW_PATH)
print(f"Raw dataset loaded: {df.shape[0]} rows, {df.shape[1]} columns")


# ----------------------------------------------------------------------------------
# 2. DUPLICATE RECORD CHECK
# ----------------------------------------------------------------------------------
section("2. DUPLICATE RECORD CHECK")

full_dupes = df.duplicated().sum()
seq_dupes = df.duplicated(subset=["order_id", "payment_sequential"]).sum()

print(f"Fully duplicated rows                              : {full_dupes}")
print(f"Duplicated (order_id + payment_sequential) combos  : {seq_dupes}")

before_rows = df.shape[0]
df = df.drop_duplicates()   # safety net; confirmed 0 found
after_rows = df.shape[0]
print(f"Rows removed as duplicates                          : {before_rows - after_rows}")
print("\nNote: repeated order_id values (multiple payment rows per order) are")
print("expected and kept -> they represent split or multi-method payments.")


# ----------------------------------------------------------------------------------
# 3. MISSING VALUE CHECK
# ----------------------------------------------------------------------------------
section("3. MISSING VALUE CHECK")

missing = df.isnull().sum()
missing = missing[missing > 0]
if missing.empty:
    print("No missing values found. No imputation needed for this file.")
else:
    print(missing)


# ----------------------------------------------------------------------------------
# 4. 'not_defined' PAYMENT TYPE HANDLING
# ----------------------------------------------------------------------------------
section("4. 'not_defined' PAYMENT TYPE HANDLING")

not_defined_count = (df["payment_type"] == "not_defined").sum()
print(f"Rows with payment_type = 'not_defined': {not_defined_count}")

if not_defined_count > 0:
    print(df.loc[df["payment_type"] == "not_defined", ["order_id", "payment_type", "payment_value"]])
    print("\nThese all coincide with payment_value = 0 -> genuinely unknown/void")
    print("transactions in the source data. Relabeling for dashboard clarity")
    print("(NOT dropped, since order_id must remain joinable to the orders table).")
    df["payment_type"] = df["payment_type"].replace({"not_defined": "unknown"})


# ----------------------------------------------------------------------------------
# 5. INVALID VALUE CHECK
# ----------------------------------------------------------------------------------
section("5. INVALID VALUE CHECK")

zero_value = (df["payment_value"] == 0).sum()
negative_value = (df["payment_value"] < 0).sum()
non_positive_installments = (df["payment_installments"] <= 0).sum()

print(f"Rows with payment_value == 0             : {zero_value} (flagged, not removed)")
print(f"Rows with payment_value < 0              : {negative_value}")
print(f"Rows with payment_installments <= 0      : {non_positive_installments} (flagged, not removed)")


# ----------------------------------------------------------------------------------
# 6. PAYMENT TYPE DISTRIBUTION (post-cleaning)
# ----------------------------------------------------------------------------------
section("6. PAYMENT TYPE DISTRIBUTION (POST-CLEANING)")
print(df["payment_type"].value_counts())


# ----------------------------------------------------------------------------------
# 7. FINAL VALIDATION
# ----------------------------------------------------------------------------------
section("7. FINAL VALIDATION")

print(f"Final shape                       : {df.shape[0]} rows, {df.shape[1]} columns")
print(f"Remaining duplicate rows          : {df.duplicated().sum()}")
print(f"Row count unchanged from raw file : {df.shape[0] == before_rows}")


# ----------------------------------------------------------------------------------
# 8. EXPORT CLEANED DATASET
# ----------------------------------------------------------------------------------
section("8. EXPORT CLEANED DATASET")

df.to_csv(CLEANED_PATH, index=False)
print(f"Cleaned dataset saved to: {CLEANED_PATH}")

section("ORDER PAYMENTS DATASET CLEANING COMPLETE")
print("This file is now safe to load into Tableau and join to orders on 'order_id'.")


 1. LOAD RAW DATA
Raw dataset loaded: 103886 rows, 5 columns

 2. DUPLICATE RECORD CHECK
Fully duplicated rows                              : 0
Duplicated (order_id + payment_sequential) combos  : 0
Rows removed as duplicates                          : 0

Note: repeated order_id values (multiple payment rows per order) are
expected and kept -> they represent split or multi-method payments.

 3. MISSING VALUE CHECK
No missing values found. No imputation needed for this file.

 4. 'not_defined' PAYMENT TYPE HANDLING
Rows with payment_type = 'not_defined': 3
                               order_id payment_type  payment_value
51280  4637ca194b6387e2d538dc89b124b0ee  not_defined            0.0
57411  00b1cb0320190ca0daa2c88b35206009  not_defined            0.0
94427  c8c528189310eaa44a745b8d9d26908b  not_defined            0.0

These all coincide with payment_value = 0 -> genuinely unknown/void
transactions in the source data. Relabeling for dashboard clarity
(NOT dropped, since order_id m